# Margin Valuation Adjustment (MVA)
## QRE-55

MVA is the present value of the funding cost of posting **initial margin (IM)** over the life of a trade. Where FVA (QRE-54) prices the funding of variation margin (driven by MTM), MVA prices the funding of initial margin — the upfront capital buffer posted to protect against gap risk during the margin period of risk.

$$\text{MVA} = s_{\text{IM}} \sum_{i=1}^{n} \Delta t_i \cdot \mathbb{E}^{\mathbb{Q}}\bigl[D(0,t_i)\,\text{IM}(t_i)\bigr]$$

where:
- $s_{\text{IM}}$ — funding spread for IM (bps/year; higher than VM since IM is often   posted in bonds/securities at a repo haircut)
- $\mathbb{E}[D(0,t_i)\,\text{IM}(t_i)]$ — **discounted Expected Initial Margin** (EIM_disc)
- $\text{IM}(t_i)$ — path-wise initial margin computed via SIMM or CCP formula

**Key structural insight:** unlike CVA and FVA, MVA depends on $\text{IM}(t)$ which is always **non-negative** — so there is no FBA-like offset. MVA is always a cost.

**Bilateral vs cleared:**

| | Bilateral SIMM | CCP (LCH/Eurex) |
|---|---|---|
| MPOR | 10 business days | 5 business days |
| IM formula | ISDA SIMM | CCP-specific (usually variance-based) |
| IM cost direction | Always a cost (post IM) | Always a cost (post IM) |
| MVA ~ | $\text{IM}_{\text{SIMM}} \propto \sqrt{10}$ | $\text{IM}_{\text{CCP}} \propto \sqrt{5}$ |


In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quant_risk.setup import base
from quant_risk.config import PROCESSED_DIR
from quant_risk.curves.ois import OISCurve
from quant_risk.models.rates import HullWhiteProcess
from quant_risk.models.simulator import MCSimulator

np, pd, plt = base()

RNG_SEED = 42


---
## 1. MVA Theory

### 1.1 What Initial Margin Is

Initial margin is a **pre-funded loss buffer** posted by both counterparties at the start of a trade and throughout its life. It is designed to cover the mark-to-market change over the **Margin Period of Risk (MPOR)** — the time it would take to close out a defaulted counterparty's portfolio.

Under BCBS-IOSCO bilateral margin rules (Phase 6 full scope):
- **MPOR = 10 business days** for bilateral uncleared OTC derivatives
- IM must cover losses at a 99% confidence level over MPOR

### 1.2 MVA Formula

The funding cost of the IM path:

$$\text{MVA} = s_{\text{IM}} \sum_{i=1}^{n} \Delta t_i \cdot \underbrace{\mathbb{E}^{\mathbb{Q}}[D(0,t_i)\,\text{IM}(t_i)]}_{\text{EIM}_{\text{disc}}(t_i)}$$

Unlike FVA (which has FCA and FBA), MVA is **unilateral** — you always pay to fund your own IM. The counterparty's IM reduces your CVA exposure but does not reduce your MVA.

The funding spread $s_{\text{IM}}$ is typically **higher than the VM spread** because:
- IM is often posted in government bonds (not cash), requiring a repo transaction
- Repo haircuts mean you post more collateral value than the IM amount
- Securities lending for IM incurs additional operational cost

Typical $s_{\text{IM}}$: 15–40 bps above OIS for IG-rated banks (vs 20–50 bps for FVA).

### 1.3 Simplified SIMM — DV01-Based IM

Under **ISDA SIMM v2.6**, the IM for an interest rate delta risk position is:

$$\text{IM}^{\text{SIMM}}(t) = \text{RW} \times |\text{DV01}_{\text{remaining}}(t)|$$

where:
- **RW (risk weight)** = prescribed sensitivity weight per tenor bucket (bps per year, embedded as a fraction below)
- **DV01**$_{\text{remaining}}(t)$ = dollar value of 1 bp on the remaining swap cashflows

$$\text{DV01}_{\text{remaining}}(t) = \frac{N}{10{,}000} \sum_{T_i > t} \delta_i P^{HW}(t, T_i; r(t)) = \frac{N}{10{,}000} \times \text{Annuity}(t)$$

The SIMM risk weights for EUR IR delta (SIMM v2.6, representative values):

| Tenor bucket | Risk weight (bps/year) |
|---|---|
| 2Y | 77 |
| 3Y | 74 |
| 5Y | 64 |
| 10Y | 60 |
| 15Y | 55 |

For simplicity this notebook uses a **single dominant tenor bucket** (the remaining maturity of the swap) and scales the risk weight interpolated from the table above.

### 1.4 MPOR Scaling

When comparing bilateral (MPOR=10) to cleared (MPOR=5), IM scales as $\sqrt{\text{MPOR}}$ under the normal distribution assumption for daily P&L:

$$\text{IM}(\text{MPOR}) = \text{IM}_{\text{base}} \times \sqrt{\frac{\text{MPOR}}{10}}$$

For cleared ($\text{MPOR}=5$): $\text{IM}_{\text{cleared}} = \text{IM}_{\text{bilateral}} / \sqrt{2}$, cutting MVA by ~29%.


In [ ]:
# ── Load OIS, build HW simulator ─────────────────────────────────────────────
try:
    ois = OISCurve.from_processed(str(PROCESSED_DIR))
    print(ois.describe())
except FileNotFoundError:
    print("Using synthetic OIS curve")
    data = pd.DataFrame(
        {"years":           [1/12,2/12,3/12,6/12,9/12,1.0,2.0,3.0,5.0,10.0,15.0],
         "zero_rate_pct":   [2.63,2.61,2.58,2.48,2.38,2.30,2.20,2.15,2.20,2.40,2.50],
         "discount_factor": [np.exp(-r/100*t) for r,t in zip(
             [2.63,2.61,2.58,2.48,2.38,2.30,2.20,2.15,2.20,2.40,2.50],
             [1/12,2/12,3/12,6/12,9/12,1,2,3,5,10,15])],
         "valuation_date":  ["2026-03-24"]*11},
        index=["1M","2M","3M","6M","9M","12M","2Y","3Y","5Y","10Y","10Y+"])
    data.index.name = "maturity"
    ois = OISCurve(data)

KAPPA, SIGMA = 0.10, 0.50
r0 = ois.forward_rate(1/12, 2/12)

sim = MCSimulator(
    process   = HullWhiteProcess(curve=ois, kappa=KAPPA, sigma=SIGMA),
    x0        = r0,
    T         = 10.0,
    n_steps   = 120,
    n_paths   = 5000,
    antithetic= True,
    seed      = RNG_SEED,
)
print(sim.describe())


In [ ]:
# ── Supporting functions — all parameters as arguments ───────────────────────

def hw_B(tau, kappa):
    return np.where(tau > 0, (1 - np.exp(-kappa * tau)) / kappa, 0.0)

def hw_bond_price_paths(t, maturities, r_t, kappa, sigma, curve):
    tau     = maturities - t
    B_tau   = hw_B(tau, kappa)
    P_0T    = np.array([curve.discount_factor(T) for T in maturities])
    P_0t    = curve.discount_factor(t) if t > 1e-6 else 1.0
    dt_fd   = 1/12; t_lo = max(t - dt_fd, dt_fd)
    f_0t    = curve.forward_rate(t_lo, t_lo + dt_fd)
    sigma_d = sigma / 100
    var_adj = (sigma_d**2 / (4*kappa)) * B_tau**2 * (1 - np.exp(-2*kappa*t))
    rate_dev = (r_t[:, None] - f_0t) / 100 * B_tau[None, :]
    return (P_0T / P_0t)[None, :] * np.exp(-rate_dev - var_adj[None, :])


def irs_annuity(
    paths: np.ndarray,
    t: float,
    payment_dates: np.ndarray,
    year_fracs: np.ndarray,
    kappa: float,
    sigma: float,
    curve: OISCurve,
    dt: float,
    n_steps: int,
) -> np.ndarray:
    """
    Remaining swap annuity at time t: A(t) = Σ_{Tᵢ>t} δᵢ P^HW(t, Tᵢ; r(t)).

    Returns
    -------
    np.ndarray : shape (n_paths,) — annuity value per path (dimensionless × time)
    """
    remaining = payment_dates[payment_dates > t]
    yf_r      = year_fracs[payment_dates > t]
    if len(remaining) == 0:
        return np.zeros(paths.shape[0])
    t_idx = min(int(round(t / dt)), n_steps)
    r_t   = paths[:, t_idx]
    P_ti  = hw_bond_price_paths(t, remaining, r_t, kappa, sigma, curve)
    return (yf_r * P_ti).sum(axis=1)


def irs_dv01(
    paths: np.ndarray,
    t: float,
    payment_dates: np.ndarray,
    year_fracs: np.ndarray,
    notional: float,
    kappa: float,
    sigma: float,
    curve: OISCurve,
    dt: float,
    n_steps: int,
) -> np.ndarray:
    """
    Dollar value of 1 basis point (DV01) of the remaining IRS at time t.

    DV01(t) = (notional / 10,000) × Annuity(t)

    This is the sensitivity of the fixed leg to a +1bp parallel shift in rates —
    a standard approximation for par swaps that holds exactly at inception.

    Parameters
    ----------
    notional : swap notional in EUR
    (all other parameters as in irs_annuity)

    Returns
    -------
    np.ndarray : shape (n_paths,) — DV01 in EUR per bp
    """
    ann = irs_annuity(paths, t, payment_dates, year_fracs,
                       kappa, sigma, curve, dt, n_steps)
    return notional / 10_000 * ann


def simm_im(
    dv01: np.ndarray,
    risk_weight_bps: float,
    mpor_days: int,
    mpor_base_days: int = 10,
) -> np.ndarray:
    """
    Simplified single-bucket SIMM IM.

    IM = RW × |DV01| × sqrt(mpor_days / mpor_base_days)

    Parameters
    ----------
    dv01            : shape (n_paths,) — DV01 in EUR per bp
    risk_weight_bps : SIMM risk weight for the dominant tenor bucket (bps)
    mpor_days       : margin period of risk in business days
    mpor_base_days  : reference MPOR for the risk weight (default 10 = SIMM bilateral)

    Returns
    -------
    np.ndarray : shape (n_paths,) — IM in EUR, always >= 0
    """
    rw       = risk_weight_bps / 10_000        # bps → decimal
    mpor_adj = np.sqrt(mpor_days / mpor_base_days)
    return rw * np.abs(dv01) * mpor_adj


def compute_mva(
    eim_disc: np.ndarray,
    exp_dates: np.ndarray,
    im_funding_spread_bps: float,
) -> dict:
    """
    MVA = s_IM × Σ Δtᵢ × EIM_disc(tᵢ)

    Parameters
    ----------
    eim_disc              : discounted expected IM, shape (n_dates,)
                            = E[D(0,t) × IM(t)] per path — computed externally
    exp_dates             : evaluation dates in years
    im_funding_spread_bps : funding spread on IM above OIS (bps/year)

    Returns
    -------
    dict with MVA scalar and per-period breakdown
    """
    s_im   = im_funding_spread_bps / 10_000
    t_prev = np.concatenate([[0.0], exp_dates[:-1]])
    dt_i   = exp_dates - t_prev

    mva_by_period = s_im * dt_i * eim_disc
    mva_total     = float(mva_by_period.sum())

    return {
        'MVA'           : mva_total,
        'eim_disc'      : eim_disc,
        'mva_by_period' : mva_by_period,
    }


In [ ]:
# ── Compute the IM profile along simulated paths ─────────────────────────────

# Trade parameters — all explicit, consistent with QRE-53/54
SWAP_MATURITY = 5.0
SWAP_COUPON   = r0
SWAP_NOTIONAL = 1_000_000
payment_dates = np.arange(1.0, SWAP_MATURITY + 0.001, 1.0)
year_fracs    = np.ones(len(payment_dates))

# SIMM parameters
SIMM_RW_BPS       = 64.0   # EUR IR delta, 5Y bucket, SIMM v2.6 representative value
MPOR_BILATERAL    = 10     # business days — BCBS-IOSCO bilateral
MPOR_CLEARED      = 5      # business days — LCH/Eurex IRS
IM_FUNDING_SPREAD = 25.0   # bps/year — IM funding spread above OIS

exp_dates = np.arange(0.5, SWAP_MATURITY + 0.5, 0.5)

# Build the discounted IM profile for each evaluation date
# For each date t:
#   1. Compute DV01(t) per path
#   2. Compute IM(t) = simm_im(DV01(t), RW, MPOR)
#   3. Discount: D(0,t) × IM(t)
#   4. EIM_disc(t) = mean across paths

eim_disc_bilateral = np.zeros(len(exp_dates))
eim_disc_cleared   = np.zeros(len(exp_dates))
eim_paths_at_dates = np.zeros((sim.n_paths, len(exp_dates)))  # for percentile plots

for j, t in enumerate(exp_dates):
    dv01_t = irs_dv01(sim.paths, t, payment_dates, year_fracs, SWAP_NOTIONAL,
                       KAPPA, SIGMA, ois, sim.dt, sim.n_steps)
    im_bilateral = simm_im(dv01_t, SIMM_RW_BPS, MPOR_BILATERAL)
    im_cleared   = simm_im(dv01_t, SIMM_RW_BPS, MPOR_CLEARED)

    sdf_t = sim.sdf(t)
    eim_disc_bilateral[j] = (sdf_t * im_bilateral).mean()
    eim_disc_cleared[j]   = (sdf_t * im_cleared).mean()
    eim_paths_at_dates[:, j] = im_bilateral

print("Discounted Expected IM profile (bilateral, MPOR=10):")
for t, eim in zip(exp_dates, eim_disc_bilateral):
    pct = eim / SWAP_NOTIONAL * 10000
    print(f"  t={t:.1f}Y  EIM_disc = {eim:>8,.2f} EUR  ({pct:.2f} bps of notional)")

print(f"\nPeak EIM_disc: {eim_disc_bilateral.max():,.2f} EUR  "
      f"at t={exp_dates[eim_disc_bilateral.argmax()]}Y")
print(f"SIMM RW={SIMM_RW_BPS:.0f} bps, MPOR={MPOR_BILATERAL} days, N={SWAP_NOTIONAL/1e6:.0f}M EUR")


In [ ]:
# ── IM profile and MVA computation ───────────────────────────────────────────

result_bilateral = compute_mva(eim_disc_bilateral, exp_dates, IM_FUNDING_SPREAD)
result_cleared   = compute_mva(eim_disc_cleared,   exp_dates, IM_FUNDING_SPREAD)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: IM profile — median, EIM_disc, and fan
ax = axes[0]
q5,  q95 = np.percentile(eim_paths_at_dates, [5, 95],  axis=0)
q25, q75 = np.percentile(eim_paths_at_dates, [25, 75], axis=0)
ax.fill_between(exp_dates, q5/1e4,  q95/1e4,  alpha=0.10, color='purple', label='5–95th pctile IM')
ax.fill_between(exp_dates, q25/1e4, q75/1e4,  alpha=0.22, color='purple', label='25–75th pctile IM')
ax.plot(exp_dates, np.median(eim_paths_at_dates, axis=0)/1e4, '-',  color='purple', lw=1.5,
        label='Median IM(t)')
ax.plot(exp_dates, eim_disc_bilateral/1e4, '--', color='black', lw=2.0,
        label='EIM_disc(t) (CVA integrand analogue)')
ax.plot(exp_dates, eim_disc_cleared/1e4,  ':',  color='steelblue', lw=1.8,
        label=f'EIM_disc cleared (MPOR={MPOR_CLEARED}d)')
ax.set_xlabel('t (years)')
ax.set_ylabel('Initial margin (EUR × 10⁴)')
ax.set_title(f'SIMM IM profile — {SWAP_MATURITY:.0f}Y IRS, N={SWAP_NOTIONAL/1e6:.0f}M EUR\n'
             f'RW={SIMM_RW_BPS:.0f} bps, σ={SIGMA}%/√yr')
ax.legend(fontsize=7)

# Right: MVA by period — waterfall
ax = axes[1]
ax.bar(exp_dates, result_bilateral['mva_by_period'],
       width=0.25, color='purple', alpha=0.8, label=f'Bilateral MPOR={MPOR_BILATERAL}d')
ax.bar(exp_dates + 0.27, result_cleared['mva_by_period'],
       width=0.25, color='steelblue', alpha=0.8, label=f'Cleared MPOR={MPOR_CLEARED}d')
ax.plot(exp_dates, result_bilateral['mva_by_period'].cumsum(), 'k--', lw=1.5,
        marker='o', ms=4, label=f'Cumulative MVA → {result_bilateral["MVA"]:,.0f} EUR')
ax.set_xlabel('Period end t (years)')
ax.set_ylabel('MVA contribution (EUR)')
ax.set_title(f'MVA by period — s_IM={IM_FUNDING_SPREAD:.0f} bps/year\n'
             f'Bilateral: {result_bilateral["MVA"]:,.2f} EUR  '
             f'Cleared: {result_cleared["MVA"]:,.2f} EUR')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"MVA Summary:")
print(f"  Bilateral (MPOR={MPOR_BILATERAL}d): MVA = {result_bilateral['MVA']:>10,.2f} EUR  "
      f"({result_bilateral['MVA']/SWAP_NOTIONAL*10000:.2f} bps)")
print(f"  Cleared   (MPOR={MPOR_CLEARED}d):  MVA = {result_cleared['MVA']:>10,.2f} EUR  "
      f"({result_cleared['MVA']/SWAP_NOTIONAL*10000:.2f} bps)")
print(f"  Clearing benefit: {(result_bilateral['MVA']-result_cleared['MVA']):>10,.2f} EUR  "
      f"(ratio = {result_bilateral['MVA']/result_cleared['MVA']:.3f} ≈ √2 = {np.sqrt(2):.3f})")


---
## 2. Cleared vs Bilateral — The MVA Impact of Clearing

### 2.1 MPOR and the √2 Rule

The IM under a central counterparty (CCP) uses MPOR=5 days instead of 10, because a CCP can close out a defaulted portfolio faster (pre-funded default fund, auction process). Under the $\sqrt{\text{MPOR}}$ scaling:

$$\frac{\text{MVA}_{\text{cleared}}}{\text{MVA}_{\text{bilateral}}} \approx \sqrt{\frac{5}{10}} = \frac{1}{\sqrt{2}} \approx 0.707$$

Clearing reduces MVA by approximately **29%** through MPOR alone.

### 2.2 Additional Clearing Costs (Partially Offsetting)

The clearing benefit is partially offset by:
- **CCP membership fees** and **default fund contributions** (not modelled here)
- **CCP IM model** may produce higher IM than SIMM at extremes (particularly for   large portfolios or stress periods)
- **IM segregation**: cleared IM is typically held by the CCP in an independently   administered account — the posting institution cannot rehypothecate it

### 2.3 Incremental MVA in Portfolio Context

When a new trade is added to a netting set, the **incremental MVA** is:

$$\Delta \text{MVA} = \text{MVA}(\text{portfolio} + \text{new trade}) - \text{MVA}(\text{portfolio})$$

Because SIMM aggregates sensitivities with correlations (not linearly), $\Delta \text{MVA}$ depends on the existing portfolio — a hedging trade (offsetting DV01) can reduce MVA to near zero.


In [ ]:
# ── MVA sensitivity: MPOR, RW, funding spread, vol ───────────────────────────

print("=== MVA Sensitivity Analysis ===\n")

# 1. MPOR comparison
print(f"1. MVA vs MPOR (RW={SIMM_RW_BPS:.0f} bps, s_IM={IM_FUNDING_SPREAD:.0f} bps):")
for mpor in [5, 7, 10, 14, 20]:
    eim_d = np.zeros(len(exp_dates))
    for j, t in enumerate(exp_dates):
        dv01_t = irs_dv01(sim.paths, t, payment_dates, year_fracs, SWAP_NOTIONAL,
                           KAPPA, SIGMA, ois, sim.dt, sim.n_steps)
        im_t   = simm_im(dv01_t, SIMM_RW_BPS, mpor)
        eim_d[j] = (sim.sdf(t) * im_t).mean()
    mva = compute_mva(eim_d, exp_dates, IM_FUNDING_SPREAD)['MVA']
    ratio_to_10 = mva / result_bilateral['MVA']
    print(f"  MPOR={mpor:2d}d  MVA={mva:>10,.2f} EUR  "
          f"ratio={ratio_to_10:.3f}  √(MPOR/10)={np.sqrt(mpor/10):.3f}")

# 2. Risk weight sensitivity
print(f"\n2. MVA vs SIMM risk weight (MPOR={MPOR_BILATERAL}d):")
for rw in [40, 50, 60, 64, 70, 80, 100]:
    eim_d = np.zeros(len(exp_dates))
    for j, t in enumerate(exp_dates):
        dv01_t = irs_dv01(sim.paths, t, payment_dates, year_fracs, SWAP_NOTIONAL,
                           KAPPA, SIGMA, ois, sim.dt, sim.n_steps)
        eim_d[j] = (sim.sdf(t) * simm_im(dv01_t, rw, MPOR_BILATERAL)).mean()
    mva = compute_mva(eim_d, exp_dates, IM_FUNDING_SPREAD)['MVA']
    print(f"  RW={rw:3.0f} bps  MVA={mva:>10,.2f} EUR  ({mva/SWAP_NOTIONAL*10000:.2f} bps)")

# 3. IM funding spread
print(f"\n3. MVA vs IM funding spread (MPOR={MPOR_BILATERAL}d, RW={SIMM_RW_BPS:.0f} bps):")
for s_im in [10, 15, 20, 25, 30, 40, 50]:
    mva = compute_mva(eim_disc_bilateral, exp_dates, s_im)['MVA']
    print(f"  s_IM={s_im:2.0f} bps  MVA={mva:>10,.2f} EUR  ({mva/SWAP_NOTIONAL*10000:.2f} bps)")

# 4. HW vol — IM grows with vol (DV01 grows as annuity widens distribution)
print(f"\n4. MVA vs HW vol σ (MPOR={MPOR_BILATERAL}d, s_IM={IM_FUNDING_SPREAD:.0f} bps):")
for sigma_v in [0.30, 0.50, 0.70, 1.00]:
    sim_v = MCSimulator(
        process  = HullWhiteProcess(curve=ois, kappa=KAPPA, sigma=sigma_v),
        x0=r0, T=10.0, n_steps=120, n_paths=5000, antithetic=True, seed=RNG_SEED,
    )
    eim_d = np.zeros(len(exp_dates))
    for j, t in enumerate(exp_dates):
        dv01_t = irs_dv01(sim_v.paths, t, payment_dates, year_fracs, SWAP_NOTIONAL,
                           KAPPA, sigma_v, ois, sim_v.dt, sim_v.n_steps)
        eim_d[j] = (sim_v.sdf(t) * simm_im(dv01_t, SIMM_RW_BPS, MPOR_BILATERAL)).mean()
    mva = compute_mva(eim_d, exp_dates, IM_FUNDING_SPREAD)['MVA']
    print(f"  σ={sigma_v:.2f}%/√yr  MVA={mva:>10,.2f} EUR  ({mva/SWAP_NOTIONAL*10000:.2f} bps)")


In [ ]:
# ── Combined XVA comparison: CVA, FVA, MVA on the same trade ─────────────────
# Bring in the CVA and FVA results from QRE-53 and QRE-54 (recompute here
# with the same parameters for a self-contained summary).

from quant_risk.models.simulator import MCSimulator as _MCSimulator

def irs_mtm(paths, t, payment_dates, year_fracs,
             K, notional, is_payer, kappa, sigma, curve, dt, n_steps):
    remaining = payment_dates[payment_dates > t]
    yf_r      = year_fracs[payment_dates > t]
    if len(remaining) == 0:
        return np.zeros(paths.shape[0])
    t_idx  = min(int(round(t / dt)), n_steps)
    r_t    = paths[:, t_idx]
    P_ti   = hw_bond_price_paths(t, remaining, r_t, kappa, sigma, curve)
    fl     = 1.0 - P_ti[:, -1]
    fx     = (K/100) * (yf_r * P_ti).sum(axis=1)
    return notional*(fl - fx) if is_payer else -notional*(fl - fx)

def mtm_fn(paths, t):
    return irs_mtm(paths, t, payment_dates, year_fracs,
                   SWAP_COUPON, SWAP_NOTIONAL, True,
                   KAPPA, SIGMA, ois, sim.dt, sim.n_steps)

profile = sim.exposure_profile(mtm_fn, exp_dates)
disc_m  = np.column_stack([sim.sdf(t) for t in exp_dates])
nee_disc = (disc_m * np.minimum(profile['mtm'], 0)).mean(axis=0)

# CVA
h       = (60/10000) / 0.60   # CDS=60bps, R=40%
t_prev  = np.concatenate([[0.0], exp_dates[:-1]])
pd_per  = np.exp(-h*t_prev) - np.exp(-h*exp_dates)
cva     = float((0.60 * profile['EE_disc'] * pd_per).sum())

# FVA
s_f, s_r = 30/10000, 20/10000
dt_i     = exp_dates - t_prev
fca = float((s_f * dt_i * profile['EE_disc']).sum())
fba = float((s_r * dt_i * np.abs(nee_disc)).sum())
fva = fca - fba

# MVA already in result_bilateral
mva = result_bilateral['MVA']

total_xva = cva + fva + mva

print("XVA Summary — 5Y Payer IRS, K=ATM, N=EUR 1M")
print(f"  CVA  = {cva:>10,.2f} EUR  ({cva/SWAP_NOTIONAL*10000:.2f} bps)  [CDS=60bps, R=40%]")
print(f"  FVA  = {fva:>10,.2f} EUR  ({fva/SWAP_NOTIONAL*10000:.2f} bps)  [s_f=30bps, s_r=20bps, no CSA]")
print(f"  MVA  = {mva:>10,.2f} EUR  ({mva/SWAP_NOTIONAL*10000:.2f} bps)  [SIMM RW=64bps, MPOR=10d, s_IM=25bps]")
print(f"  ─────────────────────────────")
print(f"  Total XVA = {total_xva:>10,.2f} EUR  ({total_xva/SWAP_NOTIONAL*10000:.2f} bps)")
print(f"  Risk-free MtM ≈ 0 (ATM at inception)")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
labels = ['CVA\n(credit)','FVA\n(funding)','MVA\n(margin)','Total XVA']
vals   = [cva, fva, mva, total_xva]
colors = ['firebrick','darkorange','purple','black']
bars = ax.bar(labels, [v/SWAP_NOTIONAL*10000 for v in vals],
               color=colors, alpha=0.8, width=0.45)
for bar, v in zip(bars, vals):
    ax.annotate(f'{v/SWAP_NOTIONAL*10000:.1f} bps',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Adjustment (bps of notional)')
ax.set_title('CVA + FVA + MVA — 5Y Payer IRS, EUR 1M\n'
             'QRE-53 × QRE-54 × QRE-55 combined')
plt.tight_layout()
plt.show()


---
## Summary

| Concept | Formula | Function |
|---|---|---|
| **DV01** | $(N/10000) \times \text{Annuity}(t)$ | `irs_dv01(paths, t, payment_dates, year_fracs, N, κ, σ, curve, dt, n_steps)` |
| **SIMM IM** | $\text{RW} \times |\text{DV01}(t)| \times \sqrt{\text{MPOR}/10}$ | `simm_im(dv01, risk_weight_bps, mpor_days)` |
| **EIM_disc** | $\mathbb{E}[D(0,t)\,\text{IM}(t)]$ | `(sim.sdf(t) * simm_im(dv01_t,...)).mean()` |
| **MVA** | $s_{\text{IM}} \sum_i \Delta t_i \cdot \text{EIM}_{\text{disc}}(t_i)$ | `compute_mva(eim_disc, dates, im_funding_spread_bps)` |

**Key insights:**
1. **MVA is always positive** — unlike FVA, there is no FBA-like offset. IM is a cost.
2. **Clearing reduces MVA by √2 (≈29%)** through MPOR reduction from 10 to 5 days
3. **MVA grows with vol** — higher σ → wider IM distribution → larger EIM_disc
4. **MVA is linear in s_IM** — doubling the IM funding spread doubles MVA
5. **DV01 collapses as swap approaches maturity** — MVA naturally decays with remaining tenor

**Combined XVA:** The total dealer adjustment for a vanilla IRS is $\text{CVA} + \text{FVA} + \text{MVA}$. For a typical EUR IG counterparty (CDS=60bps, no CSA), this is approximately 3–10 bps depending on vol and trade tenor.

**Next:** [QRE-56 — XVA Aggregation](11_xva_aggregation.ipynb) — combines CVA, DVA, FVA, and MVA under a **netting set**, applies correlation adjustments, and produces the final P&L-consistent XVA reserve under IFRS 13 / FRTB CVA framework.
